# Rust 108: Assertions and Type Checks — Practice

Companion exercises for `08_assertions-and-type-checks_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and there is one exercise for each of its demonstration cells, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the parts marked `TODO`; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. The Assertion Family: Panic as a Feature](#1-the-assertion-family-panic-as-a-feature)
- [2. Assertions at Compile Time](#2-assertions-at-compile-time)
- [3. `debug_assert!`: Checks You Can Compile Away](#3-debug_assert-checks-you-can-compile-away)
- [4. `matches!`: A Pattern Test, not a Match](#4-matches-a-pattern-test-not-a-match)
- [5. Runtime Type Inspection: `std::any`](#5-runtime-type-inspection-stdany)
- [6. Size of Things: `size_of` and Friends](#6-size-of-things-size_of-and-friends)
- [7. What Is Checkable When: Runtime vs Compile Time](#7-what-is-checkable-when-runtime-vs-compile-time)
- [8. Reading the Compiler: Panics and E-Codes](#8-reading-the-compiler-panics-and-e-codes)
- [9. Performance Notes for Low-Latency Code](#9-performance-notes-for-low-latency-code)
- [13. Review: Mixed Problems](#13-review-mixed-problems)

§1–§9 each have one exercise per demonstration cell of the example notebook; §13 mixes them. When a check fails, the fastest route back is §12 of the example notebook, which is the one-page cheat sheet.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from one cell is visible in every cell below it.
3. **Items are order-independent.** A `fn`, a `const` or a `use` line may be used above the cell that defines it.

A fourth consequence bites here exactly as in every chapter:

> **Item names must be unique across the whole notebook.** Every cell shares one namespace, so two functions with the same name are a redefinition error, even in different sections. If the compiler says a name "is defined multiple times", it is pointing at an *earlier* cell. Rename the new one.

The same one-namespace rule applies to `use` lines: this notebook imports each path **once**, at its first exercise (`std::panic` in 1.2, `std::any::type_name` in 5.1, `std::any::{Any, TypeId}` in 5.2–5.3, `std::time::Instant` in 9.1). Do not re-import per cell.

### The stub convention

Each exercise cell declares a signature, a doc comment stating the contract, and a `// --- check (do not modify) ---` block of `assert_eq!`/`assert!` calls. The body you must fill in is `todo!()`, which compiles but **panics** when executed:

```text
thread '<main>' panicked at ... not yet implemented
```

Replace the `todo!()` and re-run the cell. An `Exercise N.M passed` line means you are done.

### Three notes specific to this chapter

- **Exercise names are their own namespace.** `panic_text`, `describe_type`, `same_type` and friends exist once across the whole notebook; later exercises reuse them instead of redefining them — order-independence makes that legal.
- **`const _` items are anonymous.** Several exercises pin facts with `const _: () = assert!(..);`. Underscore items never collide — write as many as the contract needs.
- **Checks never fire on purpose.** Unlike the example notebook, no check block catches a panic: every check asserts success. The one panic-*related* exercise (1.3) hands you a payload that already exists.

### Expect warnings until you finish

A fresh notebook shows several warning families, all normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the notebook-wide imports (`std::any::type_name` in 5.1, `TypeId` in 5.2, `Instant` in 9.1) until the exercise that uses them is solved;
- `field ... is never read` on the `Fill` enum (4.1) — that comes from the *stubbed* pattern functions, which would read the fields once implemented.

Each warning disappears when you implement the body that needs it. If a diagnostic appears that is not in one of those families, it is yours.

**See**: §0 and §8 of the example notebook.


### Exercise 0.1 — Assert your first invariant

**Goal**: prove the kernel is alive by guarding a computed spread.

**Given**: nothing; this is the smoke test.

**Objectives**

- implement `checked_spread` returning `bid - ask`'s gap and asserting it is positive
- keep the signature unchanged

**See**: §0.1 and §1.1 of the example notebook.

**Hint**: compute `high - low`, `assert!(gap > 0.0, "…")` with the values in the message, then return the gap as the tail expression.


In [ ]:
/// Return `high - low` as a positive gap, asserting it is strictly positive.
///
/// TODO: replace `todo!()` with the compute-assert-return.
fn checked_spread(high: f64, low: f64) -> f64 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(checked_spread(4131.0, 4125.5), 5.5);
assert_eq!(checked_spread(100.25, 100.0), 0.25);
println!("Exercise 0.1 passed");


### Exercise 1.1 — Guard the book

**Goal**: §1.1's invariant as a function: verify both sides of a ladder before quoting.

**Objectives**

- implement `book_sane`: true only when the book is two-sided, the ladder ascends, and the best bid is below the best ask
- keep the signature unchanged

**See**: §1.1 of the example notebook.

**Hint**: three boolean checks combined with `&&`: `bids.last() < asks.first()` element-wise via iterators, and `bids.windows(2).all(|w| w[0] < w[1])` — or one `is_sorted()` call per ladder.


In [ ]:
/// True when both ladders ascend and best_bid < best_ask.
///
/// TODO: replace `todo!()` with the three boolean checks.
fn book_sane(bids: &[u64], asks: &[u64]) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(book_sane(&[4124, 4125], &[4126, 4127]));
assert!(!book_sane(&[4126], &[4125]));          // crossed
assert!(!book_sane(&[4126, 4125], &[4127]));    // bids descend
assert!(!book_sane(&[], &[4127]));              // one-sided
println!("Exercise 1.1 passed");


### Exercise 1.2 — Build the failure report yourself

**Goal**: produce `assert_eq!`'s report *shape* as a function, so its format is visible.

**Info**: the payload is ``assertion `left == right` failed\n  left: {a:?}\n  right: {b:?}`` — the two values rendered with `Debug`.

**Objectives**

- implement `eq_report` formatting both operands with `{:?}` in exactly that shape
- keep the signature unchanged

**See**: §1.2 of the example notebook.

**Hint**: `format!("assertion `left == right` failed\n  left: {a:?}\n  right: {b:?}")` — mind the two leading spaces before each value.


In [ ]:
/// Format an assert_eq!-shaped failure report from two Debug values.
///
/// TODO: replace `todo!()` with the format! call.
fn eq_report<T: std::fmt::Debug>(a: T, b: T) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(
    eq_report(4.0f64, 5.0f64),
    "assertion `left == right` failed\n  left: 4.0\n  right: 5.0"
);
assert_eq!(eq_report("a", "b").contains("left: \"a\""), true);
println!("Exercise 1.2 passed");


### Exercise 1.3 — The universal payload extractor

**Goal**: reimplement §1.3's `panic_text`: `String` first, then `&str`, then a floor.

**Objectives**

- implement `panic_text` so every payload this chapter produces becomes a `String`
- keep the signature unchanged

**See**: §1.3 of the example notebook.

**Hint**: `downcast_ref::<String>().cloned()`, `.or_else(|| downcast_ref::<&str>().map(|s| s.to_string()))`, `.unwrap_or_else(|| "<opaque payload>".to_string())`.


In [ ]:
/// Pull a human-readable message out of any panic payload.
///
/// TODO: replace `todo!()` with the two downcasts and the floor.
fn panic_text(payload: Box<dyn std::any::Any + Send>) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(panic_text(Box::new(String::from("eager"))), "eager");
assert_eq!(panic_text(Box::new("lazy")), "lazy");
assert_eq!(panic_text(Box::new(0u8)), "<opaque payload>");
println!("Exercise 1.3 passed");


### Exercise 2.1 — The numbers a const assert pins

**Goal**: produce the three size facts a layout contract would pin, so pinning them is a choice you understand.

**Info**: the cell below carries a *solved* const assert as a demo — note it asserts a fact that cannot depend on your stub, because a const assert gated on unfinished code would be a **compile error**, not a runtime failure. That is exactly why the exercise asks you to return the numbers instead: contracts come after the facts.

**Objectives**

- implement `layout_facts` returning `(size_of::<String>(), size_of::<&str>(), size_of::<Option<u32>>())`
- keep the signature unchanged

**See**: §2.1 and §6 of the example notebook.

**Hint**: `size_of` is in the prelude — no import, no turbofish beyond the type argument.


In [ ]:
// A solved const assert: true today, and a compile error the day it stops being true.
const _: () = assert!(size_of::<[u8; 32]>() == 32, "u8 array is its length in bytes");

/// Return (size of String, size of &str, size of Option<u32>) in bytes.
///
/// TODO: replace `todo!()` with the three size_of calls in a tuple.
fn layout_facts() -> (usize, usize, usize) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(layout_facts(), (24, 16, 8));
println!("Exercise 2.1 passed");


### Exercise 2.2 — Const table with pinned invariants

**Goal**: reproduce §2.2: constants, const-derived values, and a function over them.

**Objectives**

- implement `midpoint_of` computing `(first + last) / 2` of the const table `PINS`
- implement `distance_from_mid` returning `price.abs_diff(midpoint)` — one line, using both items
- keep the signatures unchanged

**See**: §2.2 of the example notebook.

**Hint**: the `const MID: u64` derivation is already written; the runtime functions just consume it.


In [ ]:
const PINS: [u64; 4] = [4124, 4125, 4126, 4127];
const MID: u64 = (PINS[0] + PINS[PINS.len() - 1]) / 2;
const _: () = assert!(PINS[0] < PINS[PINS.len() - 1], "pins must ascend");
const _: () = assert!(MID > PINS[0] && MID < PINS[PINS.len() - 1], "midpoint inside the table");

/// The midpoint of the PINS table (already const-derived as MID).
///
/// TODO: replace `todo!()` with MID.
fn midpoint_of() -> u64 {
    todo!()
}

/// Distance of `price` from the table midpoint.
///
/// TODO: replace `todo!()` with abs_diff against MID.
fn distance_from_mid(price: u64) -> u64 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(midpoint_of(), 4125);
assert_eq!(distance_from_mid(4131), 6);
assert_eq!(distance_from_mid(4119), 6);
println!("Exercise 2.2 passed");


### Exercise 3.1 — Read the build mode

**Goal**: report which build you are in, the §3.1 way.

**Objectives**

- implement `build_mode` returning `"debug"` when `cfg!(debug_assertions)` holds, `"release"` otherwise
- keep the signature unchanged

**See**: §3.1 of the example notebook.

**Hint**: `cfg!(debug_assertions)` is a plain `bool` — an `if`/`else` or a match on it is enough.


In [ ]:
/// "debug" when debug assertions are on, "release" otherwise.
///
/// TODO: replace `todo!()` with the cfg! check.
fn build_mode() -> &'static str {
    todo!()
}

// --- check (do not modify) ---
let mode = build_mode();
assert!(mode == "debug" || mode == "release");
assert_eq!(mode, if cfg!(debug_assertions) { "debug" } else { "release" });
println!("Exercise 3.1 passed (this kernel: {mode})");


### Exercise 3.2 — A debug-only guard, observed

**Goal**: the §3.2 catch, as a function: fire a `debug_assert!`, catch it, and report what happened.

**Info**: under this kernel debug assertions are **on**, so a failing `debug_assert!` panics exactly like `assert!` — and would compile to nothing in a release build.

**Objectives**

- implement `ladder_guard` so it `debug_assert!`s the ladder ascends, then returns its length
- the check block catches the panic from the deliberately broken ladder and inspects the payload with `panic_text` (exercise 1.3)
- keep the signature unchanged

**See**: §3.1, §3.2 and §1.3 of the example notebook.

**Hint**: `debug_assert!(ladder.windows(2).all(|w| w[0] < w[1]), "ladder must ascend")` — or `is_sorted()`; the body still ends with the length.


In [ ]:
/// Debug-assert the ladder ascends, then return its length.
///
/// TODO: replace `todo!()` with the debug_assert and the len.
fn ladder_guard(ladder: &[u64]) -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(ladder_guard(&[1, 2, 3]), 3);
let caught = std::panic::catch_unwind(|| ladder_guard(&[9, 8]));
let text = panic_text(caught.unwrap_err());
assert!(text.contains("ladder must ascend"), "got: {text}");
println!("Exercise 3.2 passed");


### Exercise 4.1 — `matches!` with the full pattern kit

**Goal**: §4.1's four shapes as four boolean functions.

**Objectives**

- implement the four `is_*` functions with exactly the patterns stated in their doc comments
- keep the signatures unchanged

**See**: §4.1 of the example notebook.

**Hint**: `matches!` carries rest patterns (`..`), bindings with guards (`if`), and or-patterns (`|`); `Err(_)`'s underscore is just a wildcard binding.


In [ ]:
#[derive(Debug)]
enum Fill {
    Full { qty: u32 },
    Partial { qty: u32, remaining: u32 },
}

/// True for any Partial fill, whatever its fields (rest pattern).
///
/// TODO: replace `todo!()` with the matches! call.
fn is_partial(fill: &Fill) -> bool {
    todo!()
}

/// True for a Partial fill with a positive `remaining` (binding + guard).
///
/// TODO: replace `todo!()` with the matches! call.
fn is_open(fill: &Fill) -> bool {
    todo!()
}

/// True when the result is Ok(0) or any Err (or-pattern).
///
/// TODO: replace `todo!()` with the matches! call.
fn is_rejected(res: &Result<u32, String>) -> bool {
    todo!()
}

/// True for a Full fill with qty >= 100 (binding + guard).
///
/// TODO: replace `todo!()` with the matches! call.
fn is_block(fill: &Fill) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(is_partial(&Fill::Partial { qty: 1, remaining: 1 }));
assert!(!is_partial(&Fill::Full { qty: 1 }));
assert!(is_open(&Fill::Partial { qty: 10, remaining: 2 }));
assert!(!is_open(&Fill::Partial { qty: 10, remaining: 0 }));
assert!(is_rejected(&Ok(0)));
assert!(is_rejected(&Err("no".to_string())));
assert!(!is_rejected(&Ok(7)));
assert!(is_block(&Fill::Full { qty: 300 }));
assert!(!is_block(&Fill::Full { qty: 5 }));
println!("Exercise 4.1 passed");


### Exercise 5.1 — Name a type for humans

**Goal**: §5.1's `describe` as a reusable helper.

**Objectives**

- implement `describe_type` formatting `"{n} elements of {type_name::<T>()}"` exactly
- keep the signature unchanged

**See**: §5.1 of the example notebook.

**Hint**: the format string is `"{n} elements of {}"` with `std::any::type_name::<T>()` in the hole; `n` is `values.len()`.


In [ ]:
// std::any::type_name starts here (not in the prelude).
use std::any::type_name;

/// Describe a slice as "{n} elements of {type}".
///
/// TODO: replace `todo!()` with the format! call.
fn describe_type<T>(values: &[T]) -> String {
    todo!()
}

// --- check (do not modify) ---
assert!(describe_type::<f64>(&[1.0, 2.0]).contains("2 elements of"));
assert!(describe_type::<u8>(&[]).contains("0 elements of"));
assert!(describe_type::<Vec<(String, f64)>>(&[]).contains("f64"));
println!("Exercise 5.1 passed");


### Exercise 5.2 — Identity, not description

**Goal**: §5.2's `same_type` via `TypeId`.

**Objectives**

- implement `same_type` comparing `TypeId::of::<A>()` and `TypeId::of::<B>()`
- keep the signature unchanged

**See**: §5.2 of the example notebook.

**Hint**: one comparison; both type parameters need the `'static` bound that `TypeId::of` demands.


In [ ]:
// std::any::TypeId starts here.
use std::any::TypeId;

/// True when A and B are the same type.
///
/// TODO: replace `todo!()` with the TypeId comparison.
fn same_type<A: 'static, B: 'static>() -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(same_type::<u32, u32>());
assert!(!same_type::<u32, i32>());
assert!(!same_type::<Vec<u8>, Vec<i8>>());
println!("Exercise 5.2 passed");


### Exercise 5.3 — Across the erasure boundary

**Goal**: §5.3's `identify`: ask a `dyn Any` what it really is.

**Objectives**

- implement `identify` returning `"i64: {n}"` for `i64`, `"&str: {s}"` for `&str`, `"something else"` otherwise
- keep the signature unchanged

**See**: §5.3 of the example notebook.

**Hint**: two `if let Some(..) = payload.downcast_ref::<..>()` arms in order; the fallback is the plain `else`.


In [ ]:
// std::any::Any starts here.
use std::any::Any;

/// Report what an erased value really is: "i64: {n}", "&str: {s}", or "something else".
///
/// TODO: replace `todo!()` with the two downcast arms and the fallback.
fn identify(payload: &dyn Any) -> String {
    todo!()
}

// --- check (do not modify) ---
let typed: Box<dyn Any> = Box::new(9_500_000_000i64);
let texty: Box<dyn Any> = Box::new("replay gap");
let opaque: Box<dyn Any> = Box::new([1u8, 2]);
assert_eq!(identify(&*typed), "i64: 9500000000");
assert_eq!(identify(&*texty), "&str: replay gap");
assert_eq!(identify(&*opaque), "something else");
println!("Exercise 5.3 passed");


### Exercise 6.1 — Recite the niche table

**Goal**: §6.1's table as functions — prove you know where `Option` is free.

**Objectives**

- implement the three functions returning the stated sizes
- keep the signatures unchanged

**See**: §6 of the example notebook.

**Hint**: `size_of::<Option<&i32>>()` — one call each; the niche does the work.


In [ ]:
/// Size of Option<i32> — the no-niche case, bigger than i32 itself.
///
/// TODO: replace `todo!()` with the size_of call.
fn option_int_size() -> usize {
    todo!()
}

/// Size of Option<&i32> — the niche case, equal to &i32 itself.
///
/// TODO: replace `todo!()` with the size_of call.
fn option_ref_size() -> usize {
    todo!()
}

/// Size of String — three machine words.
///
/// TODO: replace `todo!()` with the size_of call.
fn string_size() -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(option_int_size(), 8);
assert_eq!(option_ref_size(), 8);
assert_eq!(option_ref_size(), size_of::<&i32>());
assert_eq!(string_size(), 24);
println!("Exercise 6.1 passed");


### Exercise 6.2 — Handle versus payload

**Goal**: §6.2's distinction as a function over a slice view.

**Objectives**

- implement `handle_vs_payload` returning `(size_of::<&[u8]>(), size_of_val(view))`
- keep the signature unchanged

**See**: §6.2 of the example notebook.

**Hint**: `size_of_val(view)` — the function takes a reference to the value; a `&[u8]` passed as `&view[..]` or directly both work, the fat pointer carries the length.


In [ ]:
/// (size of the &[u8] handle, size of the bytes it points at).
///
/// TODO: replace `todo!()` with the two measurements.
fn handle_vs_payload(view: &[u8]) -> (usize, usize) {
    todo!()
}

// --- check (do not modify) ---
let bytes = [0u8; 10];
assert_eq!(handle_vs_payload(&bytes), (16, 10));
assert_eq!(handle_vs_payload(&bytes[..4]), (16, 4));
println!("Exercise 6.2 passed");


### Exercise 6.3 — The packing gate

**Goal**: §6.3's cache-line arithmetic as runtime functions (the const twins are §2's business).

**Objectives**

- implement `slots_per_line` returning `64 / size_of::<Slot>()`
- implement `slot_fits` returning `true` when two slots plus a 32-byte header still fit one 64-byte line
- keep the signatures unchanged

**See**: §6.3 of the example notebook.

**Hint**: `64 / size_of::<Slot>()` is the first; the second is `2 * size_of::<Slot>() + 32 <= 64`.


In [ ]:
/// A ring-buffer slot: price, qty, seq — 16 bytes on this platform.
#[derive(Debug)]
struct Slot {
    _price: u64,
    _qty: u32,
    _seq: u32,
}

/// How many Slots fit in a 64-byte cache line.
///
/// TODO: replace `todo!()` with the division.
fn slots_per_line() -> usize {
    todo!()
}

/// True when two slots plus a 32-byte header fit one 64-byte line.
///
/// TODO: replace `todo!()` with the packing inequality.
fn slot_fits() -> bool {
    todo!()
}

// --- check (do not modify) ---
let _probe = Slot { _price: 412_550, _qty: 300, _seq: 1 };
assert_eq!(slots_per_line(), 4);
assert!(slot_fits());
println!("Exercise 6.3 passed");


### Exercise 7.1 — The same invariant, both scales

**Goal**: §7.1's pair: a const-gate twin (given) and your runtime twin.

**Objectives**

- implement `ascending` with the `windows(2).all(..)` predicate
- keep the signature and the const block unchanged

**See**: §7.1 of the example notebook.

**Hint**: one iterator chain; empty and single-element slices trivially ascend.


In [ ]:
const SAMPLE_TS: [u64; 4] = [1, 4, 9, 16];
// The const twin (already written): a while loop, because windows(..).all(..)
// is not const-evaluable (E0015).
const ASCENDING: bool = {
    let mut ok = true;
    let mut i = 1;
    while i < SAMPLE_TS.len() {
        ok = ok && SAMPLE_TS[i - 1] < SAMPLE_TS[i];
        i += 1;
    }
    ok
};
const _: () = assert!(ASCENDING, "sample ts must ascend");

/// True when the slice strictly ascends (empty and single-element slices do).
///
/// TODO: replace `todo!()` with the windows(2) predicate.
fn ascending(samples: &[u64]) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(ascending(&SAMPLE_TS));
assert!(ascending(&[]));
assert!(ascending(&[5]));
assert!(!ascending(&[9, 8]));
assert!(!ascending(&[1, 1]));
println!("Exercise 7.1 passed");


### Exercise 8.1 — Name that diagnostic

**Goal**: §8's catalogue as a lookup — trigger to error code.

**Objectives**

- implement `error_code` with exactly the mapping in the doc comment
- keep the signature unchanged

**See**: §8 of the example notebook.

**Hint**: five arms: E0080, E0369+E0277, E0308, E0015, and the `unknown` catch-all.


In [ ]:
/// Map a diagnostic trigger to its code:
/// 0 = false const assert, 1 = assert_eq! on a bare struct,
/// 2 = non-bool condition in assert!, 3 = non-const method in a const block,
/// _ = "unknown".
///
/// TODO: replace `todo!()` with the five-arm match.
fn error_code(trigger: u8) -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(error_code(0), "E0080");
assert_eq!(error_code(1), "E0369 + E0277");
assert_eq!(error_code(2), "E0308");
assert_eq!(error_code(3), "E0015");
assert_eq!(error_code(9), "unknown");
println!("Exercise 8.1 passed");


### Exercise 9.1 — Measure an assert's branch

**Goal**: §9.1's experiment as data: run one loop bare and one guard-checked, report the shape.

**Info**: the ratio is machine-dependent and noisy; what the check can rely on is that **both loops compute the same sum** and both times are finite and positive.

**Objectives**

- implement `branch_cost` running `n` iterations of `acc = acc.wrapping_add(u64::from(i & 0xFF))` twice — once bare, once under `assert!(v <= 255)` — and returning `(bare, guarded, sums_equal)`
- keep the signature unchanged

**See**: §9.1 of the example notebook.

**Hint**: two `Instant::now()`/`elapsed()` pairs around the two loops; `assert_eq!` on the two sums for the boolean.


In [ ]:
// the timer, first needed here
use std::time::Instant;

/// Time the same accumulation loop bare and assert-guarded:
/// (bare duration, guarded duration, sums equal).
///
/// TODO: replace `todo!()` with the two timed loops.
fn branch_cost(n: u32) -> (std::time::Duration, std::time::Duration, bool) {
    todo!()
}

// --- check (do not modify) ---
let (bare, guarded, same) = branch_cost(1_000_000);
assert!(same);
assert!(bare > std::time::Duration::ZERO && guarded > std::time::Duration::ZERO);
println!("Exercise 9.1 passed (bare {bare:?}, guarded {guarded:?})");


## 13. Review: Mixed Problems

Two problems that chain several sections together — no new material, only assembly.

### Exercise 13.1 — The checked constructor

**Goal**: build a `CheckedPrice` that refuses out-of-range values *and* refuses bad parse text, reporting failures as strings.

**Info**: this is §7's "the type is the check" made concrete: once constructed, the value cannot be out of range, so downstream code needs no asserts.

**Objectives**

- implement `parse_price` returning `Ok(CheckedPrice)` for values in `1.0..=1_000_000.0`, `Err` otherwise
- the `Err` message must contain both the original text and the reason (`"parse"` or `"range"`)
- keep the signature and the struct unchanged

**See**: §7 and §1 of the example notebook.

**Hint**: `text.parse::<f64>()` for the first gate (`map_err(|_| format!("{text}: parse"))`); a plain `if` for the range gate with the same shape.


In [ ]:
/// A price guaranteed in range by construction — no runtime asserts downstream.
#[derive(Debug, PartialEq)]
struct CheckedPrice(f64);

/// Parse and range-check: Ok(CheckedPrice) or Err("{text}: parse|range").
///
/// TODO: replace `todo!()` with the parse and range gates.
fn parse_price(text: &str) -> Result<CheckedPrice, String> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(parse_price("4125.5"), Ok(CheckedPrice(4125.5)));
assert!(parse_price("abc").unwrap_err().contains("parse"));
assert!(parse_price("-1").unwrap_err().contains("range"));
assert!(parse_price("9999999").unwrap_err().contains("range"));
println!("Exercise 13.1 passed");


### Exercise 13.2 — Strike the contract audit

**Goal**: one function that audits a whole "record type" the way this chapter audits types: sizes, niches, and Debug-ability in one tuple.

**Info**: `size_of_val` measures the pointed-to value; the niche question is answered by comparing `size_of::<Option<&T>>()` with `size_of::<&T>()`.

**Objectives**

- implement `audit_record` returning `(size, niche_free, payload_len)` where `niche_free` is `Option<&T>` costing the same as `&T`, and `payload_len` is `size_of_val` of the given slice
- keep the signature unchanged

**See**: §6 and §5 of the example notebook.

**Hint**: three expressions: `size_of::<T>()`, `size_of::<Option<&T>>() == size_of::<&T>()`, `size_of_val(payload)`.


In [ ]:
/// Audit a record type: (size_of::<T>, Option<&T> is free, size_of_val of the payload).
///
/// TODO: replace `todo!()` with the three measurements.
fn audit_record<T: 'static>(payload: &[T]) -> (usize, bool, usize) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(audit_record::<u32>(&[1, 2, 3]), (4, true, 12));
assert_eq!(audit_record::<u8>(&[0u8; 5]).2, 5);
assert!(audit_record::<String>(&[]).0 == 24);
println!("Exercise 13.2 passed");


### What Comes Next

Chapter 08's exercises close the std-only checking toolkit. The core track has one chapter left before crates enter the picture.

| Chapter | Title | What it settles |
|---|---|---|
| **09** | **Logging** | `tracing` events and spans — the structured upgrade over the diagnostics you asserted with here |
| 10 | Structs, OOP and Traits | your own types — the `PartialEq`/`Debug` derives and newtypes this chapter kept reaching for |
| 11 | Errors in Depth | real error types for the payloads you learned to extract |

**Carry forward.** You should now be able to do these without looking anything up:

- pick between `assert!`, `assert_eq!`, `debug_assert!` and `matches!` and justify the choice
- read a const-assert failure (E0080) as a contract that stopped holding
- extract any panic payload with the two-downcast extractor
- recite the niche table and say where `Option` is free
- state which checks live at compile time, and what the const-eval subset forbids (E0015)
